## 0. Environment Setup

In [ ]:
import logging

# Route agentic_llmr debug messages to the notebook output.
# Third-party loggers (langchain, httpx, openai) stay at WARNING to avoid noise.
_handler = logging.StreamHandler()
_handler.setFormatter(logging.Formatter("%(levelname)s | %(name)s | %(message)s"))

_pkg_logger = logging.getLogger("agentic_llmr")
_pkg_logger.setLevel(logging.DEBUG)
_pkg_logger.addHandler(_handler)
_pkg_logger.propagate = False

import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=f"{Path.home()}/.env", override=True)

# ROS / ament package paths needed for PR2 xacro parsing.
# These must be set BEFORE any pycram/SDT imports.
_IAI_PR2_INSTALL = "/home/malineni/ros_packages/iai_pr2_install"
_IAI_PR2_SRC     = "/home/malineni/ros_packages/iai_pr2/iai_pr2_description"
os.environ["AMENT_PREFIX_PATH"] = _IAI_PR2_INSTALL + ":" + os.environ.get("AMENT_PREFIX_PATH", "")
os.environ["ROS_PACKAGE_PATH"]   = _IAI_PR2_SRC     + ":" + os.environ.get("ROS_PACKAGE_PATH", "")

print("Environment configured.")

In [ ]:
print("OPENAI_API_KEY configured:", bool(os.environ.get("OPENAI_API_KEY")))

In [ ]:
from langchain_openai import ChatOpenAI
from agentic_llmr.core.orchestrator import ReActAgent
from agentic_llmr.platform.world import set_active_world
from uniworld import load_pr2_apartment_world
from agentic_llmr.core.trace import render_trace

print("Imports successful.")

In [ ]:
# ROS setup
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

## 1. Load the World

We use `load_pr2_simple_world()` from `uniworld`, which builds a self-contained SDT world:
- **PR2** robot (fully articulated, 40+ joints)
- **Table** (URDF) at x=1.1 m
- **Milk carton** (STL mesh) at (0.9, 0.0, 0.85) with `Milk` semantic annotation
- **Cereal box** (STL mesh) at (0.9, 0.2, 0.85) with `Cereal` semantic annotation

In [ ]:
print("Loading PR2 simple world...")
world, robot_view, context = load_pr2_apartment_world()
set_active_world(world, robot_view)

bodies      = list(world.bodies)
annotations = list(world.semantic_annotations)

print(f"\nWorld loaded successfully!")
print(f"  Bodies              : {len(bodies)}")
print(f"  Semantic annotations: {len(annotations)}")
print("\nSemantic annotations:")
for ann in annotations:
    bodies_names = [str(getattr(b.name, "name", b.name)) for b in getattr(ann, "bodies", [])]
    print(f"  [{ann.__class__.__name__}]  bodies={bodies_names}")


In [ ]:
# world.bodies

In [ ]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")

## 2. Initialise the Agent

The `ReActAgent` (Orchestrator) is a LangGraph `StateGraph` with a five-stage pipeline:

`START → planner → supervisor ⇄ {specialists} → composer → END`

- **planner** — runs once: classifies the goal and produces an *advisory* decomposition the supervisor follows as guidance.
- **supervisor** — dispatch loop: routes to one specialist at a time with a scoped task, reusing facts already gathered.
- **specialists** (each an isolated subgraph):
  - **SceneQueryAgent** — SDT perception (poses, surfaces, relations, placement, current robot state)
  - **KinematicsAgent** — Giskard reachability, FK/IK, joint limits, self-collision, grasp poses
  - **PlanningAgent** — PyCRAM action schema discovery and physics simulation
- **composer** — runs once at the end: reasons across *all* gathered facts to frame the final answer (or an honest "can't yet, because…").

In [ ]:
llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = ReActAgent(llm=llm)

nodes = list(agent.agent_executor.get_graph().nodes.keys())
print("Orchestrator ready — planner → supervisor ⇄ specialists → composer.")
print(f"  Graph nodes : {nodes}")

print(f"Sub-agent tool breakdown:")
print(f"  SceneQueryAgent : {len(agent._scene_agent.tools)} tools")
print(f"  KinematicsAgent : {len(agent._kinematics_agent.tools)} tools")
print(f"  PlanningAgent   : {len(agent._planning_agent.tools)} tools  (schema discovery + simulation)")


## 3. Query Helper

A small utility that:
1. **Clears the scratchpad** so each query starts fresh
2. **Runs the agent** and prints the live reasoning trace
3. Returns the agent's final response

In [ ]:
import os

def run_query(instruction: str, hint: str = "") -> str:
    """Submit an instruction to the agent and display the full reasoning trace."""
    for pad in ["sdt_scratchpad.md", "giskard_scratchpad.md", "pycram_scratchpad.md"]:
        with open(pad, "w") as f:
            f.write("")

    print(f"\n{chr(9473)*65}")
    print(f"  QUERY: {instruction}")
    print(f"{chr(9473)*65}\n")

    response = agent.resolve_action(instruction=instruction, template_context=hint)

    print(f"\n{chr(9472)*65}")
    print("FINAL RESPONSE:")
    print(response[:3000] if len(response) > 3000 else response)
    print(f"{chr(9472)*65}\n")
    return response

print("Query helper ready.")


---
## Query 1 — Scene Overview

**Goal:** Get a complete picture of everything in the world.

In [ ]:
r1 = run_query(
    "Give me a complete overview of the current scene. "
    "List all objects present, their semantic types, and their 3D positions."
)

In [ ]:
# render_trace(agent.last_trace, title="Query 1 — Scene Overview",
#                output_path="outputs/q1_scene_overview.html", open_browser=False)

---
## Query 2 — Finding Objects by Type

**Goal:** Locate specific semantic categories in the scene.

In [ ]:
r2 = run_query(
    "Find all food-related objects in the scene. "
    "Report the exact 3D position of each one."
)

---
## Query 3 — Surface Query

**Goal:** Understand which objects are on a specific surface.

**Expected SDT tools:** `list_semantic_annotations`, `get_objects_on_surface`

In [ ]:
r3 = run_query(
    "Which objects are currently resting on the table? "
    "Report each object's semantic type, body name, and position."
)

In [ ]:
# render_trace(agent.last_trace, title="Query 3 — Surface query",
#                output_path="outputs/q3_surface_query.html", open_browser=False)

---
## Query 4 — Object Property Analysis

**Goal:** Inspect the physical characteristics of a specific object.

**Expected SDT tools:** `get_object_pose`, `get_object_dimensions`, `get_object_orientation`

In [ ]:
r4 = run_query(
    "Analyse the milk carton in detail: "
    "What is its exact 3D pose? What are its physical dimensions (width, depth, height)? "
    "Is it upright or lying on its side?"
)

---
## Query 5 — Spatial Relationship Reasoning

**Goal:** Understand the relative positions between objects and the robot.

**Expected SDT tools:** `get_spatial_relation`, `get_nearest_objects`

In [ ]:
r5 = run_query(
    "What is the spatial relationship between the milk carton and the cereal box? "
    "Which of the two is closer to the robot, and by how much?"
)

---
## Query 6 — Size Comparison and Disambiguation

**Goal:** Rank objects by physical size — needed to resolve instructions like *'the larger object'*.

**Expected SDT tools:** `sort_objects_by_size`, `get_object_dimensions`

In [ ]:
r6 = run_query(
    "Compare the physical sizes of the milk carton and the cereal box. "
    "Which one is larger? Provide their dimensions to justify your answer."
)

In [ ]:
# render_trace(agent.last_trace, title="Query 6 — Size Comparision",
#                output_path="outputs/q6_scene_overview.html", open_browser=False)

---
## Query 7 — Placement Planning

**Goal:** Find a suitable surface and free spots to place an object.

**Expected SDT tools:** `get_objects_on_surface`, `get_best_surface_for_placement`, `get_free_placement_spots`

In [ ]:
r7 = run_query(
    "I want to place the cereal box somewhere after picking it up. "
    "Find the best surface available and suggest at least 3 concrete (x, y, z) positions "
    "on that surface where I could place it."
)

In [ ]:
# render_trace(agent.last_trace, title="Query 7 — Surface query",
#                output_path="outputs/q7_placement.html", open_browser=False)

In [ ]:
# world.get_body_by_name('table_area_main').global_pose

---
## Query 8 — Full Pick-Up Parameter Resolution

**Goal:** Exercise the complete multi-tool reasoning pipeline for a manipulation task.

**Expected tools (in order):**
1. `write_scratchpad` — document plan
2. `list_available_actions` + `get_action_documentation` — discover `PickUpAction` schema
3. `find_objects_by_type` / `get_object_pose` — locate the milk
4. `get_object_dimensions` + `get_object_orientation` — understand object geometry
5. `check_kinematic_reachability` — verify arm can reach it
6. `get_grasp_poses` — get valid grasp descriptions

> **Note:** Execution (`simulate_action`) is intentionally skipped here. The agent will output the resolved JSON parameters.


In [ ]:
r8 = run_query(
    "Pick up the milk carton from the table.",
    hint="Resolve all parameters for a PickUpAction. Do not simulate — just output the final JSON."
)

In [ ]:
world.bodies

---
## Query 9 — Which Object First, Which Hand

**Goal:** A high-level decision. The agent must find where the objects are, reason about
whether the robot can actually reach them, and recommend an object + arm — or explain why
it can't. Nothing about tools or order is scripted; the agent decides.

In [ ]:
r9 = run_query(
    "The robot needs to pick up one of the two objects on the table. "
    "Which one is the more sensible choice to attempt first, and which hand should it use? "
    "Base your recommendation on where the objects actually are and whether the robot can reach them."
)

---
## Query 10 — Robot Configuration Inspection

**Goal:** Read the current robot joint states — useful before planning arm movements.

**Expected tools:** `get_joint_states`, `list_semantic_annotations`

In [ ]:
r10 = run_query(
    "What is the current configuration of the PR2 robot? "
    "Report the current joint positions for both arms and summarise whether the arms "
    "are in a parked, extended, or arbitrary configuration."
)

In [ ]:
r11 = run_query("does the robot hold anything?")

---
## High-Level Reasoning Queries

The queries below are deliberately high-level — they state a goal, not a recipe. The
supervisor must reason about what each goal needs, decompose it, delegate scoped sub-goals
to the right specialists, and synthesize a grounded answer (or an honest "it can't be done
yet, because…"). Nothing names a tool or prescribes an order.

**How the system is partitioned (so routing stays unambiguous):**
- **scene** — observes the world and the robot's current state (object poses, and the robot's
  own base pose / joint positions / end-effector poses / gripper state).
- **kinematics** — computes over the robot's geometry (reachability, forward & inverse
  kinematics, joint limits, self-collision). It is given the values it needs; it does not observe.
- **planning** — turns a physical command into an executable PyCRAM action.

Each capability lives in exactly one specialist, and each specialist chooses its own tools.
A goal like "can the robot reach the milk?" naturally flows **scene → kinematics**: scene
resolves where the milk is, kinematics judges reachability from the robot's real geometry.

---
## Query 12 — Reading the Arm's Current Pose

**Goal:** Introspection grounded in the robot's actual geometry. Answering "where is the
elbow vs. the hand" requires the world pose of links that aren't the base or a tracked
end-effector — so the agent must reason about how to obtain them rather than read them off.

In [ ]:
r12 = run_query(
    "How is the robot's right arm currently posed? Roughly where are its elbow and its hand "
    "in the world right now, and which of the two sits higher off the floor?"
)

---
## Query 13 — Can It Reach the Milk?

**Goal:** A grounded feasibility question. The agent must locate the milk and reason about
the robot's real reach from where it stands — then either recommend a hand or explain the
obstacle and the remedy. (The milk sits well beyond arm's length here, so the honest
answer is that the robot must reposition first — a good test of grounded reasoning.)

In [ ]:
r13 = run_query(
    "I'd like the robot to pick up the milk carton. Standing exactly where it is now, can it "
    "actually reach the milk? If it can, which hand is better suited and why? "
    "If it can't, explain what is preventing it and what would need to change first."
)

---
## Query 14 — Is a Folded Posture Self-Safe?

**Goal:** A safety question about the robot's own body. The agent must translate the
described intent ("fold the arm inward") into a concrete configuration and check it for
self-collision — deciding for itself how to express and test that posture.

In [ ]:
r14 = run_query(
    "If the robot were to fold its right arm inward across the front of its body, is there a "
    "risk it would collide with itself? Evaluate a representative folded posture and report "
    "whether any of its own parts would touch, and which ones."
)

---
## Query 15 — Near Its Limits?

**Goal:** The agent compares the arm's current configuration against its movement limits to
find the joint with the least remaining freedom. This needs both the current state and the
per-joint limits; the agent works out where each comes from and does the comparison itself.

In [ ]:
r15 = run_query(
    "Is the robot's right arm currently sitting close to any of its movement limits? "
    "Which joint has the least freedom left to move, and how much room does it have?"
)

---
## Query 16 — Full Manipulation, End to End

**Goal:** The hardest goal — a complete pick-up worked out from scratch and expressed as an
executable action, or an honest explanation of what blocks it. The agent must orchestrate
perception, kinematic feasibility, and action planning on its own, grounding every value in
the scene and reporting infeasibility truthfully rather than forcing a designator.

In [ ]:
r16 = run_query(
    "Figure out, end to end, everything the robot needs in order to pick up the cereal box, "
    "and give me the concrete action it would execute. If something prevents it from being "
    "done right now, tell me what and why instead.",
    hint="If feasible, produce a fully-resolved JSON action designator for the pick-up."
)